In [1]:
import requests
import pandas as pd

URL = "https://bank.shinhan.com/serviceEndpoint/httpDigital"

HEADERS = {
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://bank.shinhan.com",
    "Referer": "https://bank.shinhan.com/index.jsp",
    "User-Agent": "Mozilla/5.0"
}

def fetch_shinhan(date_str):
    payload = {
        "dataBody": {
            "ricInptRootInfo": {
                "serviceType": "GU",
                "serviceCode": "F1256",
                "isRule": "Y",
                "webUri": "/hpe/index.jsp"
            },
            "조회구분": "3",
            "출력구분": "3",
            "조회기준일자": date_str,
            "상품코드": "182000101"
        },
        "dataHeader": {
            "trxCd": "RSHRC0212A10",
            "language": "ko",
            "subChannel": "47",
            "channelGbn": "D0"
        }
    }

    r = requests.post(URL, json=payload, headers=HEADERS)

    try:
        return r.json()
    except:
        print("응답 오류:", r.text[:300])
        return None

In [2]:
def parse_shinhan(json_data, target_date):

    if json_data is None:
        return pd.DataFrame()

    body = json_data.get("dataBody", {})
    data = body.get("R_RIBF1256_1", [])

    rows = []

    for item in data:

        # 🔥 실제 key 확인해서 robust하게 처리
        currency = (
            item.get("통화코드")
            or item.get("통화")
            or item.get("crcyCd")
        )

        product = (
            item.get("상품명")
            or item.get("상품")
            or item.get("prdNm")
        )

        # 🔥 maturity + rate는 여러 컬럼으로 퍼져있음 → loop 필요
        for k, v in item.items():

            if v is None:
                continue

            # 🔥 금리 컬럼 패턴 잡기
            if "개월" in str(k) or "일" in str(k) or "기간" in str(k):

                maturity = k
                rate = v

                try:
                    rate = float(rate)
                except:
                    continue

                rows.append({
                    "bank": "SHINHAN",
                    "bank_code": "SHINHAN",
                    "target_date": target_date,
                    "currency": currency,
                    "maturity": maturity,
                    "rate": rate,
                    "product": product
                })

    return pd.DataFrame(rows)

In [4]:
dates = pd.date_range("2004-01-01", "2019-12-31", freq="QE")
dates = [d.strftime("%Y%m%d") for d in dates]

In [5]:
all_data = []

for i, d in enumerate(dates):

    res = fetch_shinhan(d)

    if res is None:
        continue

    if res.get("dataHeader", {}).get("result") == "FAIL":
        continue

    df = parse_shinhan(res, d)

    if not df.empty:
        all_data.append(df)

    if i % 10 == 0:
        print(i, "done")

final = pd.concat(all_data, ignore_index=True)

print("FINAL SHAPE:", final.shape)

0 done
10 done
20 done
30 done
40 done
50 done
60 done
FINAL SHAPE: (9478, 7)


In [6]:
final["target_date"] = pd.to_datetime(final["target_date"])

final = final.sort_values(["target_date", "currency", "maturity"])

# 통화 코드 정리
final["currency"] = final["currency"].astype(str).str.extract(r"([A-Z]{3})")

# maturity 공백 제거
final["maturity"] = final["maturity"].astype(str).str.replace(" ", "")

In [7]:
save_path = r"C:\Users\starw\Desktop\shinhan_2004_2019_full.xlsx"

final.to_excel(save_path, index=False)

print("저장 완료:", save_path)

저장 완료: C:\Users\starw\Desktop\shinhan_2004_2019_full.xlsx
